# Bronze — trust price history (CSV archive)

`landing.trust_prices_csv_raw` → `bronze.trust_prices_csv`. Every column cast to STRING.

Silver reads two tickers from this table, `BCPT` and `CSH` — the delisted trusts Yahoo
has erased. The rest is superseded by the Yahoo pull but still lands in full.

The known defects **must survive this layer**: the mixed date labels, the mis-scaled
tickers, and the `PCFT` zero price. Silver fixes them, and the fix is only auditable
because Bronze kept the original.

Expected: **16,357 rows**, matching Landing exactly.

In [0]:
CATALOG = "`index-vs-trust-pipeline`"
SOURCE = f"{CATALOG}.landing.trust_prices_csv_raw"
TARGET = f"{CATALOG}.bronze.trust_prices_csv"

In [0]:
src_columns = spark.table(SOURCE).columns

# Cast whatever arrived. Naming columns in advance would silently drop any the source
# adds, which is the one thing this layer must never do.
cast_list = ",\n  ".join(f"CAST(`{c}` AS STRING) AS `{c}`" for c in src_columns)
sql = f"CREATE OR REPLACE TABLE {TARGET} AS\nSELECT\n  {cast_list}\nFROM {SOURCE}"

print(f"{len(src_columns)} columns found: {src_columns}\n")
print(sql)

spark.sql(sql)
print(f"\nwrote {TARGET}")

## Verification

In [0]:
%sql
-- Row parity, plus the known-bad row that proves Bronze cleaned nothing.
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.trust_prices_csv_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv)      AS bronze_rows,
  (SELECT price_gbx_or_gbp FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
    WHERE ticker = 'PCFT' AND `date` = '2019-11-01')                            AS pcft_price;

Expect **16,357 / 16,357** and a `PCFT` price of **0.0**. A cleaned or null price here
would mean Bronze had started fixing things.

In [0]:
%sql
-- The contract. Any non-STRING column here is a bug.
SELECT COUNT(*)                                              AS columns_total,
       SUM(CASE WHEN data_type <> 'STRING' THEN 1 ELSE 0 END) AS not_string
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'trust_prices_csv';

Expect **5 columns, 0 not_string**.